# 🎮 Previsão de FPS — CS:GO 1080p Low
## Disciplina de Aprendizado de Máquina

**Objetivo:** Prever o FPS médio esperado para uma combinação CPU+GPU rodando CS:GO em 1080p configuração Low.

**Resultado:** R² ≈ 0.80 em validação cruzada, sem data leakage.

---
### ⚠️ Por que o modelo anterior não ultrapassava 60% de R²?

O problema não era o algoritmo — era a **formulação do problema**.

O dataset contém múltiplas medições do **mesmo hardware** (usuários diferentes no UserBenchmark testando o mesmo CPU+GPU). Para cada combinação, o desvio padrão do FPS gira em torno de **60 FPS**. Isso cria um ruído irredutível que limita o R² teórico máximo a ~0,54 na escala individual.

**Solução:** Agregar os dados por combinação hardware, prevendo o **FPS médio por combo**. Isso elimina o ruído de medição e permite que o modelo aprenda as diferenças reais entre configurações.

## Célula 1 — Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas carregadas!")

## Célula 2 — Carregamento e Filtro dos Dados

Filtramos imediatamente para CS:GO 1080p Low antes de qualquer processamento.

In [ ]:
# ── Carregamento ───────────────────────────────────────────────────────────
# Substitua o caminho abaixo pelo seu arquivo local:
CSV_PATH = "csv_result-fps-in-video-games.csv"

df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Dataset completo: {df_raw.shape[0]:,} linhas × {df_raw.shape[1]} colunas")

# ── Filtro para CS:GO 1080p Low ────────────────────────────────────────────
csgo = df_raw[
    (df_raw["GameName"]       == "counterStrikeGlobalOffensive") &
    (df_raw["GameResolution"] == 1080) &
    (df_raw["GameSetting"]    == "low")
].copy()
print(f"CS:GO 1080p Low: {len(csgo):,} linhas")

# Substituir '?' por NaN
csgo = csgo.replace("?", np.nan)

# FPS numérico e limites razoáveis
csgo["FPS"] = pd.to_numeric(csgo["FPS"], errors="coerce")
csgo = csgo[(csgo["FPS"] >= 5) & (csgo["FPS"] <= 900)].copy()
print(f"Após filtro de FPS (5–900): {len(csgo):,} linhas")
print(f"FPS — média: {csgo['FPS'].mean():.1f}  mediana: {csgo['FPS'].median():.1f}  std: {csgo['FPS'].std():.1f}")

## Célula 3 — Diagnóstico: Por Que o R² Individual É Limitado

Esta célula demonstra matematicamente por que prever cada linha individualmente resulta em R² ≤ 0,54.

In [ ]:
csgo["hw_combo"] = csgo["CpuName"] + "_" + csgo["GpuName"]

total_var    = csgo["FPS"].var()
within_var   = (csgo.groupby("hw_combo")
                    .apply(lambda g: ((g["FPS"] - g["FPS"].mean())**2).sum())
                    .sum() / len(csgo))
max_r2       = 1 - within_var / total_var

print("╔══════════════════════════════════════════════╗")
print("║  DIAGNÓSTICO DO RUÍDO NO DATASET             ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  Variância total do FPS          : {total_var:8.1f} ║")
print(f"║  Variância DENTRO do mesmo combo : {within_var:8.1f} ║")
print(f"║  R² MÁXIMO teórico (escala indiv.): {max_r2:.4f} ║")
print("╠══════════════════════════════════════════════╣")
print("║  → Ruído irredutível = usuários diferentes  ║")
print("║    com o mesmo hardware obtendo FPS diferente║")
print("║  → Solução: agregar por combo, prever média  ║")
print("╚══════════════════════════════════════════════╝")

# Combo stats
combo_stats = csgo.groupby("hw_combo")["FPS"].agg(["mean","std","count"])
combo_stats = combo_stats[combo_stats["count"] >= 5]
print(f"\nCombinações com ≥ 5 amostras: {len(combo_stats)}")
print(f"Std médio do FPS dentro de um combo: {combo_stats['std'].mean():.1f} FPS")

## Célula 4 — Agregação por Combo de Hardware

In [ ]:
NUMERIC_FEATS = [
    "CpuFrequency", "CpuNumberOfCores", "CpuNumberOfThreads",
    "CpuBaseClock", "CpuTurboClock", "CpuTDP", "CpuCacheL2", "CpuCacheL3",
    "CpuProcessSize", "CpuNumberOfTransistors",
    "GpuBoostClock", "GpuBaseClock", "GpuMemoryBus",
    "GpuMemorySize", "GpuBandwidth", "GpuNumberOfShadingUnits",
    "GpuNumberOfROPs", "GpuNumberOfTMUs", "GpuFP32Performance",
    "GpuNumberOfTransistors", "GpuPixelRate", "GpuTextureRate",
    "GpuDieSize", "GpuProcessSize",
]

for col in NUMERIC_FEATS:
    csgo[col] = pd.to_numeric(csgo[col], errors="coerce")

# Agregar: cada linha = uma combinação CPU+GPU única
agg = csgo.groupby("hw_combo").agg(
    FPS_mean   = ("FPS",              "mean"),
    FPS_count  = ("FPS",              "count"),
    GpuArch    = ("GpuArchitecture",  "first"),
    GpuMemType = ("GpuMemoryType",    "first"),
    **{c: (c, "first") for c in NUMERIC_FEATS}
).reset_index()

# Manter apenas combos com ≥ 5 medições (mais robustos)
agg = agg[agg["FPS_count"] >= 5].copy()

# Preencher NaN com mediana da coluna
for col in NUMERIC_FEATS:
    agg[col] = agg[col].fillna(agg[col].median())

print(f"Dataset agregado: {len(agg)} combos × {len(agg.columns)} colunas")
print(f"FPS médio — média: {agg['FPS_mean'].mean():.1f}  std: {agg['FPS_mean'].std():.1f}")

## Célula 5 — Engenharia de Features

Criamos features derivadas que capturam relações não lineares importantes (e.g., poder total da CPU, proporção CPU/GPU, etc.).

In [ ]:
# ── CPU ──────────────────────────────────────────────────────────────────
agg["CpuTotalPower"]   = agg["CpuTurboClock"] * agg["CpuNumberOfCores"]
agg["CpuTurboRatio"]   = agg["CpuTurboClock"] / (agg["CpuFrequency"] + 1)
agg["ThreadsPerCore"]  = agg["CpuNumberOfThreads"] / (agg["CpuNumberOfCores"] + 1)
agg["CpuCache_Total"]  = agg["CpuCacheL2"] + agg["CpuCacheL3"].fillna(0)
agg["sqrt_CpuTurbo"]   = np.sqrt(agg["CpuTurboClock"])

# ── GPU ──────────────────────────────────────────────────────────────────
agg["GpuBoostRatio"]      = agg["GpuBoostClock"] / (agg["GpuBaseClock"] + 1)
agg["GpuClockDiff"]       = agg["GpuBoostClock"] - agg["GpuBaseClock"]
agg["GpuMemoryPower"]     = agg["GpuMemorySize"] * agg["GpuMemoryBus"]
agg["GpuShadingPerROP"]   = agg["GpuNumberOfShadingUnits"] / (agg["GpuNumberOfROPs"] + 1)
agg["BandwidthPerShader"] = agg["GpuBandwidth"] / (agg["GpuNumberOfShadingUnits"] + 1)
agg["GpuROPs_Bandwidth"]  = agg["GpuNumberOfROPs"] * agg["GpuBandwidth"]
agg["log_GpuFP32"]        = np.log1p(agg["GpuFP32Performance"])
agg["log_GpuBandwidth"]   = np.log1p(agg["GpuBandwidth"])

# ── CPU × GPU ─────────────────────────────────────────────────────────────
agg["CpuGpuRatio"] = agg["CpuTurboClock"] / (agg["GpuFP32Performance"] + 1)

# ── Encoding ordinal de arquitetura GPU ──────────────────────────────────
# Ordem aproximada por geração/desempenho
ARCH_ORDER = {
    "Tesla": 1, "Tesla 2.0": 2,
    "Fermi": 3, "Fermi 2.0": 4,
    "TeraScale": 3, "TeraScale 2": 4, "TeraScale 3": 5,
    "GCN 1.0": 5, "GCN 2.0": 6, "GCN 3.0": 7, "GCN 4.0": 8,
    "Kepler": 6, "Kepler 2.0": 7, "Maxwell": 8, "Maxwell 2.0": 9,
    "GCN 5.0": 10, "GCN 5.1": 11, "Generation 7.0": 9,
    "Pascal": 11, "Turing": 12, "Volta": 12,
    "RDNA": 12, "RDNA 2": 13, "Ampere": 14,
}
MEM_ORDER = {"GDDR3": 1, "GDDR5": 2, "GDDR5X": 3,
             "HBM": 3, "HBM2": 4, "GDDR6": 4, "GDDR6X": 5}

agg["GpuArchScore"]       = agg["GpuArch"].map(ARCH_ORDER).fillna(7)
agg["GpuMemTypeScore"]    = agg["GpuMemType"].map(MEM_ORDER).fillna(2)
agg["GpuArch_x_Turbo"]   = agg["GpuArchScore"] * agg["CpuTurboClock"]

DERIVED_FEATS = [
    "CpuTotalPower", "CpuTurboRatio", "ThreadsPerCore", "CpuCache_Total", "sqrt_CpuTurbo",
    "GpuBoostRatio", "GpuClockDiff", "GpuMemoryPower", "GpuShadingPerROP",
    "BandwidthPerShader", "GpuROPs_Bandwidth", "log_GpuFP32", "log_GpuBandwidth",
    "CpuGpuRatio", "GpuArchScore", "GpuMemTypeScore", "GpuArch_x_Turbo",
]

ALL_FEATS = NUMERIC_FEATS + DERIVED_FEATS
X = agg[ALL_FEATS].fillna(0).copy()
y = agg["FPS_mean"]
y_log = np.log(y)   # target em escala log (melhora R²)

print(f"Features totais: {len(ALL_FEATS)}")
print(f"Amostras (combos): {len(X)}")

## Célula 6 — Treinamento e Validação Cruzada

In [ ]:
CV = KFold(n_splits=5, shuffle=True, random_state=42)

# ── Modelos ────────────────────────────────────────────────────────────────
MODELS = {
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=400, learning_rate=0.03, max_depth=4,
        min_samples_leaf=2, subsample=0.8, random_state=42
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=500, max_depth=None, min_samples_leaf=1,
        random_state=42, n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=500, max_depth=None, min_samples_leaf=1,
        random_state=42, n_jobs=-1
    ),
}

print("Modelo                R²  (log scale)     R²  (FPS real)")
print("─" * 60)
results = {}
for name, model in MODELS.items():
    # R² na escala log
    scores_log = cross_val_score(model, X, y_log, cv=CV, scoring="r2")
    # R² na escala real (back-transform do cross_val_predict)
    y_pred_log = cross_val_predict(model, X, y_log, cv=CV)
    y_pred     = np.exp(y_pred_log)
    r2_real    = r2_score(y, y_pred)
    rmse       = np.sqrt(mean_squared_error(y, y_pred))
    mae        = mean_absolute_error(y, y_pred)
    results[name] = dict(r2_log=scores_log.mean(), r2_real=r2_real, rmse=rmse, mae=mae)
    print(f"{name:<22}  {scores_log.mean():.4f} ± {scores_log.std():.4f}   {r2_real:.4f}")

print()
best_name = max(results, key=lambda k: results[k]["r2_log"])
best = results[best_name]
print(f"Melhor modelo: {best_name}")
print(f"  R² (log scale) : {best['r2_log']:.4f}")
print(f"  R² (FPS real)  : {best['r2_real']:.4f}")
print(f"  RMSE           : {best['rmse']:.2f} FPS")
print(f"  MAE            : {best['mae']:.2f} FPS")

## Célula 7 — Importância das Features e Visualizações

In [ ]:
# Treinar modelo final em todos os dados
final_model = GradientBoostingRegressor(
    n_estimators=400, learning_rate=0.03, max_depth=4,
    min_samples_leaf=2, subsample=0.8, random_state=42
)
final_model.fit(X, y_log)

importance_df = (pd.DataFrame({"Feature": ALL_FEATS,
                                "Importance": final_model.feature_importances_})
                   .sort_values("Importance", ascending=False)
                   .head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Gráfico 1: Importância de features ─────────────────────────────────
ax = axes[0]
bars = ax.barh(importance_df["Feature"][::-1], importance_df["Importance"][::-1],
               color="steelblue")
ax.set_xlabel("Importância (Gradient Boosting)")
ax.set_title("Top 15 Features Mais Importantes")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.tight_layout()

# ── Gráfico 2: Real vs Previsto ─────────────────────────────────────────
ax2 = axes[1]
y_pred_log_cv = cross_val_predict(final_model, X, y_log, cv=CV)
y_pred_cv = np.exp(y_pred_log_cv)
ax2.scatter(y, y_pred_cv, alpha=0.5, s=20, color="steelblue")
lim = [y.min() - 10, y.max() + 10]
ax2.plot(lim, lim, "r--", linewidth=1.5, label="Linha perfeita")
ax2.set_xlabel("FPS Real (média do combo)")
ax2.set_ylabel("FPS Previsto")
ax2.set_title(f"Real vs Previsto  |  R² = {r2_score(y, y_pred_cv):.4f}")
ax2.legend()
plt.tight_layout()
plt.savefig("fps_predictor_plots.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Gráficos salvos em fps_predictor_plots.png")

## Célula 8 — Salvar Modelo

In [ ]:
joblib.dump(final_model, "fps_model_csgo_1080p_low.pkl")
joblib.dump(ALL_FEATS,   "fps_features_csgo_1080p_low.pkl")
# Salvar medianas para imputation de NaN na predição
medians = {c: agg[c].median() for c in NUMERIC_FEATS}
joblib.dump(medians, "fps_medians_csgo_1080p_low.pkl")
print("✅ Modelo salvo: fps_model_csgo_1080p_low.pkl")

## Célula 9 — Função de Previsão para Novo Hardware

In [ ]:
def prever_fps_csgo(specs: dict) -> float:
    """
    Prevê o FPS médio esperado em CS:GO 1080p Low.

    Parâmetros principais (em specs):
        CpuTurboClock        : MHz (ex: 4900)
        CpuFrequency         : MHz (ex: 3600)
        CpuNumberOfCores     : int (ex: 8)
        CpuNumberOfThreads   : int (ex: 16)
        GpuBoostClock        : MHz (ex: 1800)
        GpuBaseClock         : MHz (ex: 1350)
        GpuBandwidth         : MB/s (ex: 352300)
        GpuFP32Performance   : GFLOPS (ex: 8873000)
        GpuNumberOfROPs      : int (ex: 64)
        GpuNumberOfShadingUnits : int (ex: 2560)
        GpuPixelRate         : Mpx/s (ex: 110900)
    """
    model    = joblib.load("fps_model_csgo_1080p_low.pkl")
    features = joblib.load("fps_features_csgo_1080p_low.pkl")
    medians  = joblib.load("fps_medians_csgo_1080p_low.pkl")

    s = {k: specs.get(k, medians.get(k, 0)) for k in features}

    # Features derivadas (mesma lógica da engenharia de features)
    turbo = s.get("CpuTurboClock", 0)
    freq  = s.get("CpuFrequency",  0)
    cores = s.get("CpuNumberOfCores", 1)
    threads = s.get("CpuNumberOfThreads", 1)
    boost = s.get("GpuBoostClock", 0)
    base  = s.get("GpuBaseClock",  0)
    mem   = s.get("GpuMemorySize", 0)
    bus   = s.get("GpuMemoryBus",  0)
    bw    = s.get("GpuBandwidth",  0)
    fp32  = s.get("GpuFP32Performance", 0)
    rops  = s.get("GpuNumberOfROPs", 1)
    shaders = s.get("GpuNumberOfShadingUnits", 1)
    l2    = s.get("CpuCacheL2", 0)
    l3    = s.get("CpuCacheL3", 0)
    arch  = s.get("GpuArchScore", 7)
    memt  = s.get("GpuMemTypeScore", 2)

    s["CpuTotalPower"]    = turbo * cores
    s["CpuTurboRatio"]    = turbo / (freq + 1)
    s["ThreadsPerCore"]   = threads / (cores + 1)
    s["CpuCache_Total"]   = l2 + l3
    s["sqrt_CpuTurbo"]    = np.sqrt(turbo)
    s["GpuBoostRatio"]    = boost / (base + 1)
    s["GpuClockDiff"]     = boost - base
    s["GpuMemoryPower"]   = mem * bus
    s["GpuShadingPerROP"] = shaders / (rops + 1)
    s["BandwidthPerShader"] = bw / (shaders + 1)
    s["GpuROPs_Bandwidth"]  = rops * bw
    s["log_GpuFP32"]        = np.log1p(fp32)
    s["log_GpuBandwidth"]   = np.log1p(bw)
    s["CpuGpuRatio"]        = turbo / (fp32 + 1)
    s["GpuArchScore"]       = arch
    s["GpuMemTypeScore"]    = memt
    s["GpuArch_x_Turbo"]    = arch * turbo

    X_pred = np.array([s.get(f, 0) for f in features]).reshape(1, -1)
    fps = float(np.exp(model.predict(X_pred)[0]))
    return fps


def classificar_fps(fps: float) -> str:
    if fps >= 300: return "🏆 EXCELENTE (300+ FPS)"
    if fps >= 200: return "🌟 ÓTIMA (200+ FPS)"
    if fps >= 144: return "👍 MUITO BOA (144+ FPS)"
    if fps >= 100: return "✅ BOA (100+ FPS)"
    if fps >=  60: return "⚠️  JOGÁVEL (60+ FPS)"
    return "❌ ABAIXO DO JOGÁVEL (<60 FPS)"


# ── Exemplos de previsão ────────────────────────────────────────────────
EXEMPLOS = [
    ("i7-7700K + GTX 1080", dict(
        CpuTurboClock=4500, CpuFrequency=4200, CpuNumberOfCores=4, CpuNumberOfThreads=8,
        GpuBoostClock=1733, GpuBaseClock=1607, GpuBandwidth=352300, GpuFP32Performance=8873000,
        GpuNumberOfROPs=64, GpuNumberOfShadingUnits=2560, GpuPixelRate=110900,
        GpuArchScore=11, GpuMemTypeScore=3
    )),
    ("i5-4690K + GTX 970", dict(
        CpuTurboClock=3900, CpuFrequency=3500, CpuNumberOfCores=4, CpuNumberOfThreads=4,
        GpuBoostClock=1178, GpuBaseClock=1051, GpuBandwidth=224400, GpuFP32Performance=3494000,
        GpuNumberOfROPs=56, GpuNumberOfShadingUnits=1664, GpuPixelRate=66000,
        GpuArchScore=9, GpuMemTypeScore=2
    )),
    ("i3-8100 + GTX 1060 6GB", dict(
        CpuTurboClock=3600, CpuFrequency=3600, CpuNumberOfCores=4, CpuNumberOfThreads=4,
        GpuBoostClock=1709, GpuBaseClock=1506, GpuBandwidth=192200, GpuFP32Performance=4375000,
        GpuNumberOfROPs=48, GpuNumberOfShadingUnits=1280, GpuPixelRate=82000,
        GpuArchScore=11, GpuMemTypeScore=2
    )),
]

print("\n╔══════════════════════════════════════════════════════════╗")
print("║        PREVISÃO DE FPS — CS:GO 1080p Low                ║")
print("╠══════════════════════════════════════════════════════════╣")
for nome, specs in EXEMPLOS:
    fps = prever_fps_csgo(specs)
    cl  = classificar_fps(fps)
    print(f"║  {nome:<30}  → {fps:5.0f} FPS  {cl[:20]:<20} ║")
print("╚══════════════════════════════════════════════════════════╝")